In [ ]:
from metrics import *
from utils import utility
from tslearn.metrics import SoftDTWLossPyTorch
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import scipy.signal as signal
import matplotlib.pyplot as plt

from dtw import dtw

In [ ]:
LABEL = "S_norm"

In [ ]:
def return_samples(LABEL, data):
    index = {"F_norm": [a for a in range(417,712)],
             "N_norm": [a for a in range(2,1002)],
             "Q_norm": [a for a in range(46,1133)],
             "S_norm": [a for a in range(1953, 2721)],
             "V_norm": [a for a in range(854, 1295)]}
    
    y = data[index[LABEL]]
    # y = y.reshape(y.shape[0], 1, y.shape[-1])
    
    return y
def batching(a, n):
    n = min(n, len(a))
    k, m = divmod(len(a), n)
    
    return (a[i*k+min(i, m):(i+1)*k+min(i+1, m)] for i in range(n))


def loader(label, batch_size, resample):
    data = pd.read_pickle("../data/"+label+".pkl")
    
    X = np.array(data["beat"].to_list())
    X = signal.resample(X, resample, axis=1)
    X = return_samples(label, X)
    n = len(X) // batch_size 
    a = range(n*batch_size)
    # n = int(len(data)/batch_size)-1
    
    batches = list(batching(a, n))
    X = torch.Tensor(X)
    # X = torch.cat([X, torch.zeros((X.shape[0], 512-X.shape[1]))], dim=1)
    
    return X, batches

In [ ]:
data, batches = loader(LABEL, 128, 256)
data= data.unsqueeze(1)
print(data.shape)

In [ ]:
class pipeline:
    def __init__(self, metric_size):
        self.utils = utility()
        self.dtw = SoftDTWLossPyTorch(gamma=0.01)
        self.metric_size = metric_size
        
    def process(self, input, samples):
        # Convert images once, not repeatedly
        input_imgs = self.utils.convert(input.squeeze(1))
        sample_imgs = self.utils.convert(samples.squeeze(1))

        batch_size, sample_size = input_imgs.shape[0], sample_imgs.shape[0]
        
        # Pre-allocate metrics tensor
        metrics = torch.zeros((batch_size, self.metric_size, sample_size), device=input.device)
        
        # Vectorize outer loop
        for s in range(sample_size):
            grayB = sample_imgs[s]  # Shared across batch
            # sigB = samples[s].reshape(1, -1, 1)  # Shared across batch
            sigB = samples[s]
            
            # Parallelize batch computation
            grayA = input_imgs  # All input images
            # sigA = input.reshape(batch_size, -1, 1)  # All input signals
            sigA = input
            
            # Precompute DTW for the entire batch
            # dtw_values = torch.stack([self.dtw(sigA[b].reshape(1,-1,1), sigB) for b in range(batch_size)])
            tmp = []
            for b in range(batch_size):
                align = dtw(sigA[b], sigB, dist_method='euclidean')
                tmp.append(torch.tensor(align.distance))
            dtw_values = torch.stack(tmp)
            
            # Compute all metrics for the batch
            metric_results = torch.stack([torch.tensor([UQI.process(grayA[b], grayB),
                                          VIFP.process(grayA[b], grayB),
                                          SCC.process(grayA[b], grayB),
                                          SAM.process(grayA[b], grayB),
                                          ERGAS.process(grayA[b], grayB),
                                          RASE.process(grayA[b], grayB),
                                          SIFT.process(grayA[b], grayB),
                                          SSIM.process(grayA[b], grayB),
                                          dtw_values[b]]) for b in range(batch_size)], dim=0)
            
            metrics[:, :, s] = metric_results

        # Take the mean across the sample dimension
        # metrics = metrics.mean(dim=2)
        return metrics
    
pip = pipeline(9)

In [ ]:
# REAL DATA TO BE COMPARED WITH
REAL = data[np.random.randint(low = 0,high=data.shape[0], size=4)]

In [ ]:
# Load real and generated Q signals => Real comparison

# a = data[np.random.randint(low = 0,high=data.shape[0], size=4)]
gen = data[np.random.randint(low = 0,high=data.shape[0], size=16)]


# # visualize these signals

# plt.figure()
# plt.plot(a.squeeze(1).transpose(0,1))
# plt.plot(b.squeeze(1).transpose(0,1))
# plt.savefig("tmp.png")

# metrics = pip.process(a,b)

summ = defaultdict(list)
for j in range(4):
    # a = data[np.random.randint(low = 0, high = data.shape[0], size = 4)]
    b = gen[(j*4):(4 + j*4)]
    
    metrics = pip.process(REAL,b)
    summ["UQI"] += torch.Tensor.tolist(metrics[:,0].reshape(1,-1))[0]
    summ["VIFP"] += torch.Tensor.tolist(metrics[:,1].reshape(1,-1))[0]
    summ["SCC"] += torch.Tensor.tolist(metrics[:,2].reshape(1,-1))[0]
    summ["SAM"] += torch.Tensor.tolist(metrics[:,3].reshape(1,-1))[0]
    summ["ERGAS"] += torch.Tensor.tolist(metrics[:,4].reshape(1,-1))[0]
    summ["RASE"] += torch.Tensor.tolist(metrics[:,5].reshape(1,-1))[0]
    summ["SIFT"] += torch.Tensor.tolist(metrics[:,6].reshape(1,-1))[0]
    summ["SSIM"] += torch.Tensor.tolist(metrics[:,7].reshape(1,-1))[0]
    summ["DTW"] += torch.Tensor.tolist(metrics[:,8].reshape(1,-1))[0]

summ = pd.DataFrame(summ)
print("Metrics of Real signals")
print(summ.mean())
print(summ.min())
print(summ.max())

In [ ]:
# Load real and generated Q signals => Transformers 1d

# a = data[np.random.randint(low = 0,high=data.shape[0], size=4)]
gen = torch.load("../Transformer/"+LABEL[0]+"_1d.pt")


# # visualize these signals

# plt.figure()
# plt.plot(a.squeeze(1).transpose(0,1))
# plt.plot(b.squeeze(1).transpose(0,1))
# plt.savefig("tmp.png")

# metrics = pip.process(a,b)

summ = defaultdict(list)
for j in range(4):
    # a = data[np.random.randint(low = 0, high = data.shape[0], size = 4)]
    b = gen[(j*4):(4 + j*4)]
    
    metrics = pip.process(REAL,b)
    summ["UQI"] += torch.Tensor.tolist(metrics[:,0].reshape(1,-1))[0]
    summ["VIFP"] += torch.Tensor.tolist(metrics[:,1].reshape(1,-1))[0]
    summ["SCC"] += torch.Tensor.tolist(metrics[:,2].reshape(1,-1))[0]
    summ["SAM"] += torch.Tensor.tolist(metrics[:,3].reshape(1,-1))[0]
    summ["ERGAS"] += torch.Tensor.tolist(metrics[:,4].reshape(1,-1))[0]
    summ["RASE"] += torch.Tensor.tolist(metrics[:,5].reshape(1,-1))[0]
    summ["SIFT"] += torch.Tensor.tolist(metrics[:,6].reshape(1,-1))[0]
    summ["SSIM"] += torch.Tensor.tolist(metrics[:,7].reshape(1,-1))[0]
    summ["DTW"] += torch.Tensor.tolist(metrics[:,8].reshape(1,-1))[0]

summ = pd.DataFrame(summ)
print("Metrics of Transformer 1d generated signals")
print(summ.mean())
print(summ.min())
print(summ.max())

In [ ]:
# Load real and generated Q signals => Unet 1d

# a = data[np.random.randint(low = 0,high=data.shape[0], size=4)]
gen = torch.load("../Unet/"+LABEL[0]+"_1d.pt")


# # visualize these signals

# plt.figure()
# plt.plot(a.squeeze(1).transpose(0,1))
# plt.plot(b.squeeze(1).transpose(0,1))
# plt.savefig("tmp.png")

# metrics = pip.process(a,b)

summ = defaultdict(list)
for j in range(4):
    # a = data[np.random.randint(low = 0, high = data.shape[0], size = 4)]
    b = gen[(j*4):(4 + j*4)]
    
    metrics = pip.process(REAL,b)
    summ["UQI"] += torch.Tensor.tolist(metrics[:,0].reshape(1,-1))[0]
    summ["VIFP"] += torch.Tensor.tolist(metrics[:,1].reshape(1,-1))[0]
    summ["SCC"] += torch.Tensor.tolist(metrics[:,2].reshape(1,-1))[0]
    summ["SAM"] += torch.Tensor.tolist(metrics[:,3].reshape(1,-1))[0]
    summ["ERGAS"] += torch.Tensor.tolist(metrics[:,4].reshape(1,-1))[0]
    summ["RASE"] += torch.Tensor.tolist(metrics[:,5].reshape(1,-1))[0]
    summ["SIFT"] += torch.Tensor.tolist(metrics[:,6].reshape(1,-1))[0]
    summ["SSIM"] += torch.Tensor.tolist(metrics[:,7].reshape(1,-1))[0]
    summ["DTW"] += torch.Tensor.tolist(metrics[:,8].reshape(1,-1))[0]

summ = pd.DataFrame(summ)
print("Metrics of Unet1d generated signals")
print(summ.mean())
print(summ.min())
print(summ.max())

In [ ]:
# Load real and generated Q signals => UnetAttn

# a = data[np.random.randint(low = 0,high=data.shape[0], size=4)]
gen = torch.load("../UnetAttn/"+LABEL[0]+"_1d.pt")

# # visualize these signals

# plt.figure()
# plt.plot(a.squeeze(1).transpose(0,1))
# plt.plot(b.squeeze(1).transpose(0,1))
# plt.savefig("tmp.png")

# metrics = pip.process(a,b)

summ = defaultdict(list)
for j in range(4):
    # a = data[np.random.randint(low = 0, high = data.shape[0], size = 4)]
    b = gen[(j*4):(4 + j*4)]
    
    metrics = pip.process(REAL,b)
    summ["UQI"] += torch.Tensor.tolist(metrics[:,0].reshape(1,-1))[0]
    summ["VIFP"] += torch.Tensor.tolist(metrics[:,1].reshape(1,-1))[0]
    summ["SCC"] += torch.Tensor.tolist(metrics[:,2].reshape(1,-1))[0]
    summ["SAM"] += torch.Tensor.tolist(metrics[:,3].reshape(1,-1))[0]
    summ["ERGAS"] += torch.Tensor.tolist(metrics[:,4].reshape(1,-1))[0]
    summ["RASE"] += torch.Tensor.tolist(metrics[:,5].reshape(1,-1))[0]
    summ["SIFT"] += torch.Tensor.tolist(metrics[:,6].reshape(1,-1))[0]
    summ["SSIM"] += torch.Tensor.tolist(metrics[:,7].reshape(1,-1))[0]
    summ["DTW"] += torch.Tensor.tolist(metrics[:,8].reshape(1,-1))[0]

summ = pd.DataFrame(summ)
print("Metrics of Unet1d Attn generated signals")
print(summ.mean())
print(summ.min())
print(summ.max())